[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeteoSwiss/opendata-nwp-demos/blob/main/02_download_soil_temp.ipynb)
[![launch - renku](https://renkulab.io/renku-badge.svg)](https://renkulab.io/p/meteoswiss/opendata-nwp-demos/sessions/01KME52HC2FZ6ZHB30SSFG08PW/start)

# Downloading ICON Forecast Data via Python API
This notebook guides you through the download process of a forecast file from the ICON-CH1-EPS numerical weather model using [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/), designed to simplify working with numerical weather model data and MeteoSwiss' earthkit-data plugin [earthkit-data-meteoswiss-opendata](https://meteoswiss.github.io/earthkit-data-meteoswiss-opendata/), which enables data access to NWP MeteoSwiss Open Data.

The data you will access is provided by MeteoSwiss as part of Switzerland’s Open Government Data (OGD) initiative.

---

## 🔍 **What You’ll Do in This Notebook**

⬇️  **Download Forecast Files**  
    Retrieve and save the requested ICON-CH1-EPS forecast files (e.g. soil temperature) to disk for offline storage using earthkit-data's `to_target()` function.

📄  **Inspect Local Forecast Files**  
    Read and inspect the downloaded ICON-CH1-EPS forecast file with earthkit-data tools.

---

##  ⬇️ Downloading Forecast Files
In this first part, we retrieve ICON-CH1-EPS soil temperature forecast data from the [STAC (SpatioTemporal Asset Catalog) API](https://data.geo.admin.ch/api/stac/static/spec/v1/apitransactional.html#tag/Data/operation/getAsset), which provides structured access to Switzerland’s open geospatial data

#### 📁  Browsing the STAC Catalog (Optional)

If you'd like to explore the ICON-CH1/2-EPS forecast datasets interactively before downloading, you can browse them directly in the STAC catalog:

&nbsp;&nbsp;&nbsp;&nbsp;🔗  [Browse the ICON-CH1-EPS collection](https://data.geo.admin.ch/browser/#/collections/ch.meteoschweiz.ogd-forecasting-icon-ch1?.language=en)

&nbsp;&nbsp;&nbsp;&nbsp;🔗  [Browse the ICON-CH2-EPS collection](https://data.geo.admin.ch/browser/#/collections/ch.meteoschweiz.ogd-forecasting-icon-ch2?.language=en)


Below is a screenshot of the ICON-CH2-EPS collection as seen in the STAC browser interface.

![browser-ch2.png](./images/browser-ch2.png)

⚙️ Notebook Setup

This cell installs the required dependencies when running the notebook in Google Colab or RenkuLab.

It is skipped in a local Jupyter environment, where dependencies are assumed to be installed already.

In [ ]:
# 📦 Notebook setup: Colab + RenkuLab
import sys, os, pathlib

IN_COLAB = "google.colab" in sys.modules
IN_RENKU = "RENKU_BASE_URL" in os.environ or "RENKU_BASE_URL_PATH" in os.environ

if IN_COLAB:
    !git clone https://github.com/MeteoSwiss/opendata-nwp-demos.git
    %cd opendata-nwp-demos

if IN_COLAB or IN_RENKU:
    !pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi

    venv = pathlib.Path(".venv")
    site = venv / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    sys.path.insert(0, str(site))
    os.environ["ECCODES_DEFINITION_PATH"] = str((venv / "share/eccodes-cosmo-resources/definitions").resolve())

⚙️ Definition Setup

MeteoSwiss uses custom ecCodes definitions for ICON data. To correctly interpret MeteoSwiss-specific `shortName` values, you must configure the definition path by running the cell below.

In [ ]:
import os
import eccodes_cosmo_resources

os.environ["ECCODES_DEFINITION_PATH"] = str(
    eccodes_cosmo_resources.get_definitions_path()
)

os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

### Creating a Request
To retrieve ICON-CH1-EPS soil temperature (`T_SO`), we first define an API request. The request targets lead time zero - the forecast's initialization time.

>⏰ **Forecast Availability**: Forecast data will typically be available a couple of hours after the reference time — due to the model runtime and subsequent upload time. The data remains accessible for 24 hours after upload.

In [ ]:
import datetime as dt

request = {
    "collection": "ogd-forecasting-icon-ch1",
    "variable": "T_SO",
    "ref_time": "latest",
    "perturbed": False,
    "lead_time": dt.timedelta(hours=0)
}

Each argument in the request serves the following purpose:

| Argument             | Description |
|----------------------|-------------|
| `collection`         | Forecast collection to use (e.g., `ogd-forecasting-icon-ch1`). |
| `variable`           | Meteorological variable of interest (`T_SO` = soil temperature). |
| `ref_time` | Initialization time of the forecast in **UTC**, provided as either:<br>- The string `"latest"` to select the newest forecast run (`ref_time`) that contains all requested lead times. All assets returned by a single request therefore share the same `ref_time`. Be cautious: separate requests (for example, for different variables) resolve `"latest"` independently and may return different `ref_time` values while MeteoSwiss is uploading a new forecast run. <br>- [datetime.datetime](https://docs.python.org/3/library/datetime.html#datetime-objects) object (e.g.,<br> &nbsp; `datetime.datetime(2025, 5, 22, 9, 0, 0, tzinfo=datetime.timezone.utc)`) <br>- [ISO 8601](https://en.wikipedia.org/wiki/ISO_8601#Combined_date_and_time_representations) date string (e.g., `"2025-05-22T09:00:00Z"`)|
| `perturbed`          | If `True`, retrieves ensemble forecast members; if `False`, returns the deterministic forecast. |
| `lead_time`            | Forecast lead time, provided as either:<br>– [datetime.timedelta](https://docs.python.org/3/library/datetime.html#timedelta-objects) object (e.g., `datetime.timedelta(hours=0)`) <br>– [ISO 8601](https://en.wikipedia.org/wiki/ISO_8601#Durations) duration string (e.g., `"P0DT0H"`)|

### Downloading Data
We now send our request to the API and retrieve the resulting dataset using the `earthkit-data-meteoswiss-opendata` plugin.
The response is returned as an **[earthkit.data.data.grib.GribData](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/data/grib/index.html#earthkit.data.data.grib.GribDatal)**, which represents GRIB data, a standardized, compact file format for meteorological data. <br>
In a second step we use the function [to_target()](https://earthkit-data.readthedocs.io/en/latest/concepts/targets/to_target.html#to_target) to store the forecast file locally. The function expects a path to a directory, where the data will be stored. In this example the forecast file is stored in the folder `forecast_files/` in the current directory.

> 💡 **Tip**: Use temporary caching with earthkit-data to skip repeated downloads — it's auto-cleaned after the session.
> *For more details, see the [earthkit-data caching docs](https://earthkit-data.readthedocs.io/en/latest/examples/cache.html)*.

> 💡 **Hint**: If you get an error message containing `HTTPError: 403 Client Error: Forbidden for url`, you may be trying to retrieve data older than 24h hours! Please adjust your requests.

In [ ]:
from pathlib import Path
import earthkit.data as ekd
ekd.config.set("cache-policy", "temporary")

# Define the target directory for saving the forecast files
target_dir = Path.cwd() / "forecast_files"
target_dir.mkdir(parents=True, exist_ok=True)

forecast = ekd.from_source(
    "meteoswiss-opendata",
    **request,
    )

forecast.to_target("file", target_dir / "soil_temperature.grib")


After saving the file you can list the content of the `forecast_files` directory.

In [ ]:
# List all downloaded files in the target directory
print("\n Downloaded files:")
for file in sorted(target_dir.iterdir()):
    print(f" - {file.name}")

## 📄 Inspect Local Forecast Files
After the download, use earthkit-data's `from_source()` function to load the local GRIB file and inspect it with `ls()`.

In [ ]:
soil_temperature = ekd.from_source("file", "forecast_files/soil_temperature.grib")
fieldlist = soil_temperature.to_fieldlist()
fieldlist.ls()